In [1]:
!mamba install pandas

mambajs 0.21.4

Specs: xeus-python, numpy, matplotlib, pillow, ipywidgets>=8.1.6, ipyleaflet, scipy, pandas
Channels: emscripten-forge-4x, conda-forge

Solving environment...
Solving took 3.371 seconds
  Name    Version  Build                Channel
-------------------------------------------------------------
+ pandas  3.1.0    np23py313h124dbc7_0  emscripten-forge-4x
- pip     26.2.2   pyh145f28c_0         conda-forge


In [2]:
import pandas as pd
import numpy as np
import os

print("Библиотеки успешно подключены!")

Библиотеки успешно подключены!


In [4]:
!mamba install openpyxl

mambajs 0.21.4

Specs: xeus-python, numpy, matplotlib, pillow, ipywidgets>=8.1.6, ipyleaflet, scipy, pandas, openpyxl
Channels: emscripten-forge-4x, conda-forge

Solving environment...
Solving took 0.761 seconds
  Name        Version  Build         Channel
--------------------------------------------------
+ et_xmlfile  2.0.0    pyhd8ed1ab_1  conda-forge
+ openpyxl    3.0.9    pyhd8ed1ab_0  conda-forge


In [12]:
order_files = [
    "data/orders_2024_01.csv",
    "data/orders_2024_02.csv",
    "data/orders_2024_03.csv"
]

df_orders = pd.concat(
    [
        pd.read_csv(file, parse_dates=["order_datetime"])
        for file in order_files
    ],
    ignore_index=True
)

print("Размер таблицы заказов:", df_orders.shape)

display(df_orders.head())
df_orders.info()

Размер таблицы заказов: (15000, 7)


,order_id,user_id,order_datetime,product_id,price,quantity,status
0,1,1044,2024-01-03 18:39:26,26,14038.15,1,refunded
1,2,1079,2024-01-03 17:45:43,16,9172.60,1,paid
2,3,1041,2024-01-06 22:46:38,25,9035.26,2,paid
3,4,1083,2024-01-08 21:03:51,9,15234.33,3,paid
4,5,1068,2024-01-03 23:26:53,24,4134.38,2,paid


<class 'pandas.DataFrame'>
RangeIndex: 15000 entries, 0 to 14999
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   order_id        15000 non-null  int64         
 1   user_id         15000 non-null  int64         
 2   order_datetime  15000 non-null  datetime64[us]
 3   product_id      15000 non-null  int64         
 4   price           15000 non-null  float64       
 5   quantity        15000 non-null  int64         
 6   status          15000 non-null  str           
dtypes: datetime64[us](1), float64(1), int64(4), str(1)
memory usage: 761.8 KB


In [13]:
df_products = pd.read_csv("data/products.csv")
df_categories = pd.read_csv("data/categories.csv")

print("Таблица товаров:")
display(df_products.head())

print("Таблица категорий:")
display(df_categories.head())

Таблица товаров:


,product_id,product_name,category_id,category_name,weight_kg
0,1,Product 1,1,Electronics,11.55
1,2,Product 2,3,Sport,2.22
2,3,Product 3,5,Toys,11.43
3,4,Product 4,1,Electronics,4.17
4,5,Product 5,5,Toys,1.81


Таблица категорий:


,category_id,category_name,description
0,1,Electronics,Description for Electronics
1,2,Home,Description for Home
2,3,Sport,Description for Sport
3,4,Fashion,Description for Fashion
4,5,Toys,Description for Toys


In [14]:
df_prod_full = df_products.merge(
    df_categories,
    on="category_id",
    how="left",
    suffixes=("", "_cat")
)

# Если название категории отсутствовало в таблице товаров,
# используем название из таблицы категорий
if "category_name_cat" in df_prod_full.columns:
    df_prod_full["category_name"] = (
        df_prod_full["category_name"]
        .fillna(df_prod_full["category_name_cat"])
    )

print("Объединённая таблица товаров и категорий:")
display(df_prod_full.head())

print("Размер таблицы:", df_prod_full.shape)

Объединённая таблица товаров и категорий:


,product_id,product_name,category_id,category_name,weight_kg,category_name_cat,description
0,1,Product 1,1,Electronics,11.55,Electronics,Description for Electronics
1,2,Product 2,3,Sport,2.22,Sport,Description for Sport
2,3,Product 3,5,Toys,11.43,Toys,Description for Toys
3,4,Product 4,1,Electronics,4.17,Electronics,Description for Electronics
4,5,Product 5,5,Toys,1.81,Toys,Description for Toys


Размер таблицы: (30, 7)


In [15]:
df_orders_prod = df_orders.merge(
    df_prod_full,
    on="product_id",
    how="left"
)

print("Итоговая таблица заказов с товарами и категориями:")
display(df_orders_prod.head())

print("Размер итоговой таблицы:", df_orders_prod.shape)

Итоговая таблица заказов с товарами и категориями:


,order_id,user_id,order_datetime,product_id,price,quantity,status,product_name,category_id,category_name,weight_kg,category_name_cat,description
0,1,1044,2024-01-03 18:39:26,26,14038.15,1,refunded,Product 26,3,Sport,0.74,Sport,Description for Sport
1,2,1079,2024-01-03 17:45:43,16,9172.60,1,paid,Product 16,3,Sport,1.29,Sport,Description for Sport
2,3,1041,2024-01-06 22:46:38,25,9035.26,2,paid,Product 25,3,Sport,0.20,Sport,Description for Sport
3,4,1083,2024-01-08 21:03:51,9,15234.33,3,paid,Product 9,5,Toys,3.94,Toys,Description for Toys
4,5,1068,2024-01-03 23:26:53,24,4134.38,2,paid,Product 24,1,Electronics,0.56,Electronics,Description for Electronics


Размер итоговой таблицы: (15000, 13)


In [16]:
df_orders_prod["revenue"] = (
    df_orders_prod["price"] * df_orders_prod["quantity"]
)

category_revenue = (
    df_orders_prod
    .groupby("category_name")
    .agg(
        revenue_sum=("revenue", "sum"),
        orders_count=("order_id", "nunique"),
        items_count=("quantity", "sum")
    )
    .sort_values("revenue_sum", ascending=False)
)

print("Выручка по категориям:")
display(category_revenue)

Выручка по категориям:


,revenue_sum,orders_count,items_count
category_name,,,
Toys,1.160671e+08,4544,11337
Sport,1.003407e+08,3900,9680
Electronics,7.742044e+07,3023,7637
Fashion,5.196834e+07,2002,5048
Home,3.874713e+07,1531,3863


In [17]:
top_5_categories = category_revenue.head(5)

print("Топ-5 категорий по выручке:")
display(top_5_categories)

Топ-5 категорий по выручке:


,revenue_sum,orders_count,items_count
category_name,,,
Toys,1.160671e+08,4544,11337
Sport,1.003407e+08,3900,9680
Electronics,7.742044e+07,3023,7637
Fashion,5.196834e+07,2002,5048
Home,3.874713e+07,1531,3863


In [18]:
top_5_categories.to_csv(
    "data/top_5_categories.csv",
    encoding="utf-8-sig"
)

In [19]:
user_metrics = (
    df_orders_prod
    .groupby("user_id")
    .agg(
        orders_count=("order_id", "nunique"),
        items_count=("quantity", "sum"),
        revenue_sum=("revenue", "sum"),
        revenue_mean=("revenue", "mean"),
        first_order=("order_datetime", "min"),
        last_order=("order_datetime", "max")
    )
    .reset_index()
)

print("Показатели пользователей:")
display(user_metrics.head())

Показатели пользователей:


,user_id,orders_count,items_count,revenue_sum,revenue_mean,first_order,last_order
0,1001,177,454,4435767.09,25060.831017,2024-01-02 02:51:37,2024-03-31 21:32:19
1,1002,123,281,2830004.52,23008.166829,2024-01-02 12:57:34,2024-03-31 19:45:16
2,1003,156,389,4025266.18,25802.988333,2024-01-02 02:02:44,2024-03-31 19:36:26
3,1004,147,380,3788030.56,25768.915374,2024-01-01 00:19:34,2024-03-31 20:01:40
4,1005,141,346,3185653.37,22593.286312,2024-01-01 01:11:51,2024-03-31 18:16:10


In [20]:
df_events_records = pd.read_json(
    "data/events_records.json",
    orient="records"
)

df_events_lines = pd.read_json(
    "data/events_lines.json",
    lines=True
)

df_events = pd.concat(
    [df_events_records, df_events_lines],
    ignore_index=True
)

df_events["event_time"] = pd.to_datetime(
    df_events["event_time"]
)

print("Таблица событий:")
display(df_events.head())

print("Размер таблицы событий:", df_events.shape)

Таблица событий:


,0,1,2,3,4,5,6,7,8,9,...,1995,1996,1997,1998,1999,event_id,user_id,event_time,event_type,page
0,1,2,3,4,5,6,7,8,9,10,...,1996,1997,1998,1999,2000,NaN,NaN,NaT,NaN,NaN
1,1085,1006,1024,1062,1012,1037,1087,1081,1098,1093,...,1065,1093,1051,1057,1081,NaN,NaN,NaT,NaN,NaN
2,2024-01-30T20:37:32,2024-02-24T22:55:13,2024-02-14T17:49:43,2024-02-16T11:21:32,2024-02-08T05:25:21,2024-02-21T19:57:26,2024-02-23T20:13:14,2024-01-07T04:27:34,2024-02-14T12:37:11,2024-02-17T19:51:23,...,2024-02-28T08:23:07,2024-01-19T17:02:37,2024-02-03T15:46:28,2024-02-21T16:34:39,2024-01-15T16:12:39,NaN,NaN,NaT,NaN,NaN
3,page_view,add_to_cart,click,click,add_to_cart,add_to_cart,add_to_cart,scroll,scroll,page_view,...,click,scroll,page_view,page_view,click,NaN,NaN,NaT,NaN,NaN
4,/product/26,/product/28,/product/27,/product/1,/,/product/19,/product/4,/product/9,/product/6,/product/12,...,/product/23,/product/2,/product/5,/,/,NaN,NaN,NaT,NaN,NaN


Размер таблицы событий: (2005, 2005)


In [21]:
events_by_user = (
    df_events
    .groupby("user_id")
    .agg(
        events_total=("event_id", "count"),
        first_event=("event_time", "min"),
        last_event=("event_time", "max")
    )
    .reset_index()
)

print("Количество событий по пользователям:")
display(events_by_user.head())

Количество событий по пользователям:


,user_id,events_total,first_event,last_event
0,1001.0,29,2024-02-01 20:18:15,2024-03-31 22:29:02
1,1002.0,25,2024-02-05 07:27:49,2024-03-31 21:24:08
2,1003.0,21,2024-02-09 07:00:19,2024-03-29 20:53:24
3,1004.0,22,2024-02-03 21:50:28,2024-03-27 04:58:11
4,1005.0,21,2024-02-02 13:55:51,2024-03-30 04:58:21


In [22]:
user_features = user_metrics.merge(
    events_by_user,
    on="user_id",
    how="outer"
)

print("Объединённая таблица user_features:")
display(user_features.head())

print("Размер таблицы:", user_features.shape)

Объединённая таблица user_features:


,user_id,orders_count,items_count,revenue_sum,revenue_mean,first_order,last_order,events_total,first_event,last_event
0,1001,177,454,4435767.09,25060.831017,2024-01-02 02:51:37,2024-03-31 21:32:19,29,2024-02-01 20:18:15,2024-03-31 22:29:02
1,1002,123,281,2830004.52,23008.166829,2024-01-02 12:57:34,2024-03-31 19:45:16,25,2024-02-05 07:27:49,2024-03-31 21:24:08
2,1003,156,389,4025266.18,25802.988333,2024-01-02 02:02:44,2024-03-31 19:36:26,21,2024-02-09 07:00:19,2024-03-29 20:53:24
3,1004,147,380,3788030.56,25768.915374,2024-01-01 00:19:34,2024-03-31 20:01:40,22,2024-02-03 21:50:28,2024-03-27 04:58:11
4,1005,141,346,3185653.37,22593.286312,2024-01-01 01:11:51,2024-03-31 18:16:10,21,2024-02-02 13:55:51,2024-03-30 04:58:21


Размер таблицы: (100, 10)


In [23]:
user_cat = (
    df_orders_prod
    .groupby(["user_id", "category_name"])
    .agg(
        category_revenue=("revenue", "sum")
    )
    .reset_index()
)

top_cat_per_user = (
    user_cat
    .sort_values(
        ["user_id", "category_revenue"],
        ascending=[True, False]
    )
    .drop_duplicates("user_id")
    .rename(
        columns={
            "category_name": "top_category",
            "category_revenue": "top_category_revenue"
        }
    )
)

print("Самые доходные категории пользователей:")
display(top_cat_per_user.head())

Самые доходные категории пользователей:


,user_id,top_category,top_category_revenue
4,1001,Toys,1238283.37
9,1002,Toys,902481.68
14,1003,Toys,1448523.72
18,1004,Sport,1111600.01
24,1005,Toys,1021461.33


In [24]:
user_features = user_features.merge(
    top_cat_per_user[
        ["user_id", "top_category", "top_category_revenue"]
    ],
    on="user_id",
    how="left"
)

user_features["top_category_revenue_share"] = (
    user_features["top_category_revenue"]
    / user_features["revenue_sum"]
)

print("Пользовательские признаки с долей выручки:")
display(
    user_features[
        [
            "user_id",
            "revenue_sum",
            "top_category",
            "top_category_revenue",
            "top_category_revenue_share"
        ]
    ].head(10)
)

Пользовательские признаки с долей выручки:


,user_id,revenue_sum,top_category,top_category_revenue,top_category_revenue_share
0,1001,4435767.09,Toys,1238283.37,0.279159
1,1002,2830004.52,Toys,902481.68,0.318898
2,1003,4025266.18,Toys,1448523.72,0.359858
3,1004,3788030.56,Sport,1111600.01,0.293451
4,1005,3185653.37,Toys,1021461.33,0.320644
5,1006,4160640.89,Toys,1513177.28,0.363689
6,1007,4064170.75,Toys,1214503.49,0.298832
7,1008,2886592.62,Toys,1064621.64,0.368816
8,1009,3545781.73,Sport,1027725.41,0.289845
9,1010,4240012.44,Toys,1203500.00,0.283844


In [25]:
print("Информация о таблице user_features:")
user_features.info()

print("\nКоличество пропусков:")
display(user_features.isna().sum())

print("\nПервые 10 строк:")
display(user_features.head(10))

Информация о таблице user_features:
<class 'pandas.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 13 columns):
 #   Column                      Non-Null Count  Dtype         
---  ------                      --------------  -----         
 0   user_id                     100 non-null    int64         
 1   orders_count                100 non-null    int64         
 2   items_count                 100 non-null    int64         
 3   revenue_sum                 100 non-null    float64       
 4   revenue_mean                100 non-null    float64       
 5   first_order                 100 non-null    datetime64[us]
 6   last_order                  100 non-null    datetime64[us]
 7   events_total                100 non-null    int64         
 8   first_event                 100 non-null    datetime64[us]
 9   last_event                  100 non-null    datetime64[us]
 10  top_category                100 non-null    str           
 11  top_category_revenue        100 no

user_id                       0
orders_count                  0
items_count                   0
revenue_sum                   0
revenue_mean                  0
first_order                   0
last_order                    0
events_total                  0
first_event                   0
last_event                    0
top_category                  0
top_category_revenue          0
top_category_revenue_share    0
dtype: int64


Первые 10 строк:


,user_id,orders_count,items_count,revenue_sum,revenue_mean,first_order,last_order,events_total,first_event,last_event,top_category,top_category_revenue,top_category_revenue_share
0,1001,177,454,4435767.09,25060.831017,2024-01-02 02:51:37,2024-03-31 21:32:19,29,2024-02-01 20:18:15,2024-03-31 22:29:02,Toys,1238283.37,0.279159
1,1002,123,281,2830004.52,23008.166829,2024-01-02 12:57:34,2024-03-31 19:45:16,25,2024-02-05 07:27:49,2024-03-31 21:24:08,Toys,902481.68,0.318898
2,1003,156,389,4025266.18,25802.988333,2024-01-02 02:02:44,2024-03-31 19:36:26,21,2024-02-09 07:00:19,2024-03-29 20:53:24,Toys,1448523.72,0.359858
3,1004,147,380,3788030.56,25768.915374,2024-01-01 00:19:34,2024-03-31 20:01:40,22,2024-02-03 21:50:28,2024-03-27 04:58:11,Sport,1111600.01,0.293451
4,1005,141,346,3185653.37,22593.286312,2024-01-01 01:11:51,2024-03-31 18:16:10,21,2024-02-02 13:55:51,2024-03-30 04:58:21,Toys,1021461.33,0.320644
5,1006,161,410,4160640.89,25842.490000,2024-01-01 12:14:12,2024-03-30 15:02:30,15,2024-02-15 06:51:33,2024-03-30 07:16:02,Toys,1513177.28,0.363689
6,1007,157,394,4064170.75,25886.437898,2024-01-01 00:21:30,2024-03-31 22:01:48,18,2024-02-02 16:58:38,2024-03-22 00:22:36,Toys,1214503.49,0.298832
7,1008,122,288,2886592.62,23660.595246,2024-01-01 09:23:25,2024-03-27 12:47:15,22,2024-02-01 17:03:26,2024-03-31 15:49:24,Toys,1064621.64,0.368816
8,1009,147,366,3545781.73,24120.964150,2024-01-02 01:42:13,2024-03-31 22:34:31,19,2024-02-01 23:39:55,2024-03-31 03:00:24,Sport,1027725.41,0.289845
9,1010,155,388,4240012.44,27354.918968,2024-01-02 06:18:34,2024-03-31 21:19:30,15,2024-02-09 16:26:54,2024-03-30 02:24:55,Toys,1203500.00,0.283844


In [26]:
user_features.to_csv(
    "data/user_features_variant1.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Файл user_features_variant1.csv успешно сохранён!")

Файл user_features_variant1.csv успешно сохранён!
